# Lab 3: Advanced Spark & Production Patterns

**DAT535 - Data Engineering**

This lab continues directly from **Lab 2 (Spark Fundamentals & the Medallion Architecture)** and
reuses the **exact same Silver-layer dataset** that Lab 2 produced and saved to
`~/spark-lab-data/shared/silver/events`. Complete Lab 2 first if you have not already done so.

## Learning Objectives

- Use **window functions** for ranking, lag/lead, running totals and moving averages
- Choose effective **partitioning strategies** and understand partition pruning
- Apply **caching/persistence** correctly and know the storage-level trade-offs
- Perform every **join type** (inner, left, right, full, semi, anti) and optimize joins with broadcast
- Compare **UDFs / pandas UDFs** against built-in functions
- Build a minimal **Structured Streaming** pipeline with a windowed aggregation
- Apply production patterns: incremental processing, data-quality checks, SCD Type 2, safe aggregation


In [ ]:
# Setup and Imports
import os
import time
from datetime import datetime, timedelta

import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, when, count, sum as spark_sum, avg,
    round as spark_round, desc, asc, unix_timestamp,
    countDistinct, lag, lead, row_number, rank, dense_rank, ntile,
    broadcast, udf, pandas_udf, window as tumbling_window
)
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, StringType
from pyspark import StorageLevel

print("Libraries imported successfully")


In [ ]:
# SparkSession tuned with performance-related configuration
spark = SparkSession.builder \
    .appName("DAT535-Lab3-AdvancedSpark") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.parquet.compression.codec", "snappy") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print(f"Spark Version: {spark.version}")
print(f"UI: {spark.sparkContext.uiWebUrl}")
print("Adaptive Query Execution: enabled")


## Part 0: Load the Shared Dataset Produced by Lab 2

We deliberately do **not** regenerate the data here. Reusing the Silver-layer output from Lab 2
means both labs work with an identical dataset, which mirrors how a real pipeline is composed of
independently-runnable stages that read each other's output.


In [ ]:
SHARED_DIR = os.path.expanduser("~/spark-lab-data/shared")
SILVER_PATH = f"{SHARED_DIR}/silver/events"
BASE_DIR = os.path.expanduser("~/spark-lab-data/lab3")
os.makedirs(BASE_DIR, exist_ok=True)

if not os.path.exists(SILVER_PATH):
    raise FileNotFoundError(
        f"Could not find {SILVER_PATH}. Please run Lab 2 (lab2_spark_fundamentals.ipynb) first - "
        "it generates and saves the shared dataset used by this lab."
    )

events_df = spark.read.parquet(SILVER_PATH)
print(f"Loaded {events_df.count():,} Silver events from Lab 2")
events_df.printSchema()


## Part 1: Window Functions & Advanced Analytics

Window functions operate on a set of rows related to the current row, enabling powerful analytics
without expensive self-joins.

```
function(column).over(Window.partitionBy(...).orderBy(...))
                       |                      |
          groups rows like GROUP BY   orders rows inside
          but keeps every row         each partition
```


In [ ]:
print("=== Window Functions: Ranking ===")

purchases_df = events_df.filter(
    (col("event_type") == "purchase") & (col("total_amount").isNotNull())
)
print(f"Total purchases for analysis: {purchases_df.count():,}")

user_window = Window.partitionBy("user_id").orderBy(desc("total_amount"))

ranked_purchases = purchases_df.select(
    col("user_id"), col("event_id"), col("total_amount"), col("category"),
    row_number().over(user_window).alias("row_num"),
    rank().over(user_window).alias("rank"),
    dense_rank().over(user_window).alias("dense_rank"),
    ntile(4).over(user_window).alias("quartile"),
)

sample_user = ranked_purchases.select("user_id").first()["user_id"]
print(f"--- Ranking demo for user {sample_user} ---")
ranked_purchases.filter(col("user_id") == sample_user).orderBy("row_num").show(10)


In [ ]:
print("--- Top 3 purchases per user ---")
top_purchases_per_user = ranked_purchases.filter(col("row_num") <= 3).orderBy("user_id", "row_num")
top_purchases_per_user.show(15)


### 1.2 Analytic Functions: LAG and LEAD

In [ ]:
print("=== Window Functions: LAG and LEAD ===")

session_window = Window.partitionBy("user_id").orderBy("event_timestamp")

user_activity = events_df.select(
    col("user_id"), col("event_timestamp"), col("event_type"), col("device"),
    lag("event_type", 1).over(session_window).alias("prev_event"),
    lag("event_timestamp", 1).over(session_window).alias("prev_timestamp"),
    lead("event_type", 1).over(session_window).alias("next_event"),
).withColumn(
    "seconds_since_prev",
    when(col("prev_timestamp").isNotNull(),
         unix_timestamp(col("event_timestamp")) - unix_timestamp(col("prev_timestamp")))
    .otherwise(None)
)

sample_user = user_activity.select("user_id").first()["user_id"]
user_activity.filter(col("user_id") == sample_user) \
    .select("event_timestamp", "event_type", "prev_event", "next_event", "seconds_since_prev") \
    .orderBy("event_timestamp").show(10)


In [ ]:
print("--- What events lead to a purchase? ---")
user_activity.filter(col("event_type") == "purchase") \
    .groupBy("prev_event").agg(count("*").alias("num_purchases")) \
    .orderBy(desc("num_purchases")).show()


### 1.3 Running Totals and Moving Averages

In [ ]:
print("=== Running Totals & Moving Averages ===")

daily_sales = purchases_df.groupBy("event_date").agg(
    spark_sum("total_amount").alias("daily_revenue"),
    count("*").alias("num_orders"),
).orderBy("event_date")

date_window = Window.orderBy("event_date")
rolling_7day = Window.orderBy("event_date").rowsBetween(-6, 0)

daily_metrics = daily_sales \
    .withColumn("cumulative_revenue", spark_sum("daily_revenue").over(date_window)) \
    .withColumn("7day_avg_revenue", spark_round(avg("daily_revenue").over(rolling_7day), 2)) \
    .withColumn(
        "day_over_day_change",
        spark_round(
            (col("daily_revenue") - lag("daily_revenue", 1).over(date_window)) /
            lag("daily_revenue", 1).over(date_window) * 100, 2)
    )

daily_metrics.show(15)


In [ ]:
print("=== User Lifetime Value Progression ===")

user_purchase_window = Window.partitionBy("user_id").orderBy("event_timestamp")

user_ltv = purchases_df.select(
    col("user_id"), col("event_timestamp"), col("total_amount"),
    spark_sum("total_amount").over(user_purchase_window).alias("lifetime_value"),
    count("*").over(user_purchase_window).alias("total_purchases"),
    spark_round(avg("total_amount").over(user_purchase_window), 2).alias("avg_order_value"),
)

sample_user = user_ltv.select("user_id").first()["user_id"]
print(f"User {sample_user}'s LTV progression:")
user_ltv.filter(col("user_id") == sample_user).orderBy("event_timestamp").show()


## Part 2: Partitioning Strategies

In [ ]:
print(f"Current DataFrame partitions: {events_df.rdd.getNumPartitions()}")
partition_base = f"{BASE_DIR}/partitioned_data"


In [ ]:
print("--- Strategy 1: No Partitioning ---")
no_partition_path = f"{partition_base}/no_partition"

start_time = time.time()
events_df.coalesce(4).write.mode("overwrite").parquet(no_partition_path)
print(f"Write time: {time.time() - start_time:.2f}s")

sample_date = events_df.select("event_date").first()["event_date"]
start_time = time.time()
result = spark.read.parquet(no_partition_path).filter(col("event_date") == sample_date).count()
print(f"Query time (full scan): {time.time() - start_time:.4f}s, result: {result}")


In [ ]:
print("--- Strategy 2: Partition by Date ---")
date_partition_path = f"{partition_base}/date_partition"

start_time = time.time()
events_df.write.mode("overwrite").partitionBy("event_date").parquet(date_partition_path)
print(f"Write time: {time.time() - start_time:.2f}s")

start_time = time.time()
result = spark.read.parquet(date_partition_path).filter(col("event_date") == sample_date).count()
print(f"Query time (partition pruning): {time.time() - start_time:.4f}s, result: {result}")


In [ ]:
print("--- Strategy 3: Multi-Level Partitioning ---")
multi_partition_path = f"{partition_base}/multi_partition"

start_time = time.time()
events_df.write.mode("overwrite").partitionBy("event_date", "country").parquet(multi_partition_path)
print(f"Write time: {time.time() - start_time:.2f}s")

sample_country = events_df.select("country").first()["country"]
start_time = time.time()
result = spark.read.parquet(multi_partition_path) \
    .filter((col("event_date") == sample_date) & (col("country") == sample_country)).count()
print(f"Query time (multi-partition pruning): {time.time() - start_time:.4f}s, result: {result}")


In [ ]:
print("""
PARTITIONING BEST PRACTICES
----------------------------
DO:
  - Partition by columns used in WHERE clauses (date, country, etc.)
  - Keep partition count manageable (< 10,000 partitions)
  - Target partition file sizes of 128MB - 1GB

DON'T:
  - Partition by high-cardinality columns (user_id, event_id)
  - Over-partition (too many nested partition levels)
  - Create many tiny files
""")


## Part 3: Caching & Memory Management

In [ ]:
complex_df = events_df \
    .filter(col("event_type").isin(["purchase", "add_to_cart", "page_view"])) \
    .withColumn("is_purchase", when(col("event_type") == "purchase", 1).otherwise(0)) \
    .withColumn("hour_bucket",
                when(col("event_hour") < 6, "night")
                .when(col("event_hour") < 12, "morning")
                .when(col("event_hour") < 18, "afternoon")
                .otherwise("evening"))

print(f"complex_df rows: {complex_df.count():,}")


In [ ]:
print("--- Without Caching (recomputed on every action) ---")
start_time = time.time()
q1 = complex_df.groupBy("category").agg(count("*")).collect()
q2 = complex_df.groupBy("device").agg(spark_sum("is_purchase")).collect()
q3 = complex_df.groupBy("hour_bucket").agg(avg("price")).collect()
no_cache_time = time.time() - start_time
print(f"Total time (3 actions, no cache): {no_cache_time:.2f}s")


In [ ]:
print("--- With Caching (computed once, reused) ---")
cached_df = complex_df.cache()

start_time = time.time()
q1 = cached_df.groupBy("category").agg(count("*")).collect()   # triggers the cache
cache_time = time.time() - start_time
print(f"First action (includes caching): {cache_time:.4f}s")

start_time = time.time()
q2 = cached_df.groupBy("device").agg(spark_sum("is_purchase")).collect()
q3 = cached_df.groupBy("hour_bucket").agg(avg("price")).collect()
subsequent_time = time.time() - start_time
print(f"Subsequent actions (from cache): {subsequent_time:.4f}s")

total_cached = cache_time + subsequent_time
print(f"Total with caching: {total_cached:.2f}s | Speedup vs no-cache: {no_cache_time/total_cached:.1f}x")


In [ ]:
print("""
STORAGE LEVEL OPTIONS
----------------------------------------------------------------
MEMORY_ONLY (default) : fast, but limited by available memory
MEMORY_AND_DISK        : spills to disk when memory is full
MEMORY_ONLY_SER        : serialized - less memory, more CPU
MEMORY_AND_DISK_SER    : serialized + spill - best for large datasets
DISK_ONLY              : slowest, but never runs out of memory
""")

large_cached_df = events_df.persist(StorageLevel.MEMORY_AND_DISK)
large_cached_df.count()
print("Cached with MEMORY_AND_DISK storage level")

cached_df.unpersist()
large_cached_df.unpersist()


## Part 4: Joins - All Join Types & Broadcast Optimization

Joins are one of the most expensive operations in Spark because they usually require a **shuffle**
(unless one side is small enough to **broadcast**).


In [ ]:
category_details = spark.createDataFrame([
    ("Electronics", "Tech", 0.08),
    ("Clothing", "Fashion", 0.05),
    ("Books", "Media", 0.0),
    ("Home", "Lifestyle", 0.06),
    ("Sports", "Active", 0.05),
    ("Beauty", "Personal", 0.07),
], ["category", "department", "tax_rate"])

print(f"Category dimension table: {category_details.count()} rows")
category_details.show()


In [ ]:
purchase_events = events_df.filter(
    (col("event_type") == "purchase") & (col("category").isNotNull())
)

print("--- inner join: only categories present in BOTH tables ---")
purchase_events.join(category_details, on="category", how="inner").select(
    "event_id", "category", "department", "tax_rate").show(5)

print("--- left (outer) join: keep all purchase rows, nulls if no match ---")
purchase_events.join(category_details, on="category", how="left").select(
    "event_id", "category", "department").show(5)

print("--- right join: keep all dimension rows, even categories with 0 purchases ---")
purchase_events.join(category_details, on="category", how="right").groupBy("category").count().show()

print("--- full outer join: keep everything from both sides ---")
purchase_events.join(category_details, on="category", how="full").select(
    "event_id", "category").show(5)

print("--- left semi join: like a filter - keep purchase rows that HAVE a match, no new columns ---")
purchase_events.join(category_details, on="category", how="left_semi").show(5)

print("--- left anti join: keep purchase rows that have NO match (useful for finding orphans) ---")
purchase_events.join(category_details, on="category", how="left_anti").show(5)


In [ ]:
print("--- Optimization: Broadcast Joins for small dimension tables ---")

start_time = time.time()
regular_join = purchase_events.join(category_details, on="category", how="left")
regular_count = regular_join.count()
regular_time = time.time() - start_time
print(f"Regular join: {regular_time:.4f}s, rows: {regular_count}")

start_time = time.time()
broadcast_join = purchase_events.join(broadcast(category_details), on="category", how="left")
broadcast_count = broadcast_join.count()
broadcast_time = time.time() - start_time
print(f"Broadcast join: {broadcast_time:.4f}s, rows: {broadcast_count}")
print(f"Speedup: {regular_time/broadcast_time:.1f}x (broadcast avoids shuffling the large side)")


## Part 5: Query Optimization

In [ ]:
print("--- Filter early vs filter late ---")

print("Anti-pattern: filter AFTER an expensive join")
start_time = time.time()
bad_query = events_df \
    .join(broadcast(category_details), on="category", how="left") \
    .filter(col("country") == "US") \
    .filter(col("event_type") == "purchase") \
    .count()
bad_time = time.time() - start_time
print(f"Late filter: {bad_time:.4f}s, rows: {bad_query}")

print("Best practice: filter BEFORE the join to shrink the data early")
start_time = time.time()
good_query = events_df \
    .filter((col("country") == "US") & (col("event_type") == "purchase")) \
    .join(broadcast(category_details), on="category", how="left") \
    .count()
good_time = time.time() - start_time
print(f"Early filter: {good_time:.4f}s, rows: {good_query}")


In [ ]:
print("--- Column pruning: select only what you need, as early as possible ---")

start_time = time.time()
all_cols = events_df.filter(col("event_type") == "purchase") \
    .groupBy("category").agg(spark_sum("total_amount").alias("revenue")).collect()
print(f"Without pruning: {time.time() - start_time:.4f}s")

start_time = time.time()
few_cols = events_df.select("event_type", "category", "total_amount") \
    .filter(col("event_type") == "purchase") \
    .groupBy("category").agg(spark_sum("total_amount").alias("revenue")).collect()
print(f"With column pruning: {time.time() - start_time:.4f}s")


In [ ]:
print("--- Reading the physical execution plan ---")
optimized_query = events_df \
    .select("event_type", "category", "total_amount", "country") \
    .filter((col("event_type") == "purchase") & (col("country") == "US")) \
    .groupBy("category").agg(spark_sum("total_amount").alias("revenue"))

optimized_query.explain(True)


## Part 6: UDFs vs Built-in Functions

Built-in Spark SQL functions run inside the JVM and are optimized by Catalyst. Python UDFs require
serializing every row to a Python process and back, which is much slower. Pandas UDFs (vectorized)
sit in between: they batch rows into pandas Series using Arrow, which is far faster than row-at-a-time
Python UDFs.


In [ ]:
print("--- Built-in function (fastest, preferred whenever possible) ---")

start_time = time.time()
builtin_result = purchase_events.withColumn(
    "price_tier",
    when(col("total_amount") >= 200, "high")
    .when(col("total_amount") >= 50, "medium")
    .otherwise("low")
).count()
builtin_time = time.time() - start_time
print(f"Built-in when/otherwise: {builtin_time:.4f}s")


In [ ]:
print("--- Row-at-a-time Python UDF (slower: per-row Python <-> JVM serialization) ---")

def price_tier_udf(amount):
    if amount is None:
        return "unknown"
    if amount >= 200:
        return "high"
    if amount >= 50:
        return "medium"
    return "low"

price_tier = udf(price_tier_udf, StringType())

start_time = time.time()
udf_result = purchase_events.withColumn("price_tier", price_tier(col("total_amount"))).count()
udf_time = time.time() - start_time
print(f"Python UDF: {udf_time:.4f}s")


In [ ]:
print("--- Pandas (vectorized) UDF: batches rows via Arrow, much faster than a row UDF ---")
import pandas as pd

@pandas_udf(DoubleType())
def apply_discount(amount: pd.Series) -> pd.Series:
    return amount * 0.9

start_time = time.time()
pandas_udf_result = purchase_events.withColumn(
    "discounted_amount", apply_discount(col("total_amount"))
).count()
pandas_udf_time = time.time() - start_time
print(f"Pandas UDF: {pandas_udf_time:.4f}s")

print("""
RULE OF THUMB:
  1. Prefer built-in functions (col, when, cast, ...) - fastest, fully optimized
  2. Use pandas UDFs when you need custom logic that vectorizes over a batch
  3. Use row-at-a-time Python UDFs only as a last resort
""")


## Part 7: Structured Streaming Basics

Structured Streaming lets you write the **same DataFrame code** against an unbounded stream of data.
Here we simulate a stream by replaying our (already-generated) events as a rate-limited micro-batch
source, and compute a windowed aggregation - a very common data-engineering task (e.g. "revenue per
5-minute window").


In [ ]:
# A `rate` source produces a self-incrementing stream - useful for demos without external systems
stream_df = spark.readStream.format("rate").option("rowsPerSecond", 5).load()

# Attach a tumbling event-time window over the streaming timestamp column
windowed_counts = stream_df \
    .withWatermark("timestamp", "10 seconds") \
    .groupBy(tumbling_window(col("timestamp"), "5 seconds")) \
    .agg(count("*").alias("event_count"))

query = windowed_counts.writeStream \
    .outputMode("update") \
    .format("memory") \
    .queryName("windowed_stream_demo") \
    .start()

time.sleep(12)  # let a few micro-batches run
query.stop()

print("Windowed streaming results captured in the in-memory sink 'windowed_stream_demo':")
spark.sql("SELECT * FROM windowed_stream_demo ORDER BY window").show(truncate=False)


In [ ]:
print("""
In a real pipeline you would replace the `rate` source with:
  - spark.readStream.format("kafka")...            (message queue)
  - spark.readStream.format("parquet")/"json"...   (files landing in a directory - "file source")
and the `memory` sink with a durable sink such as:
  - .format("parquet").option("path", ...).option("checkpointLocation", ...)
  - .format("kafka")...

The DataFrame transformation logic (window, groupBy, agg) stays IDENTICAL between batch and
streaming - this is Structured Streaming's core idea: one API for both.
""")


## Part 8: Production Patterns

In [ ]:
print("--- Pattern 1: Incremental Processing (only process new data since the last run) ---")

last_processed = "2024-01-15T00:00:00"
incremental_df = events_df.filter(col("event_timestamp") > last_processed)

total = events_df.count()
incremental = incremental_df.count()
print(f"Total records: {total:,}")
print(f"New records since {last_processed}: {incremental:,} ({incremental/total*100:.1f}%)")


In [ ]:
print("--- Pattern 2: Data Quality Framework ---")

def run_quality_checks(df, table_name, critical_columns):
    print(f"\nQuality report for: {table_name}")
    total_rows = df.count()
    print(f"Total rows: {total_rows:,}")

    existing_cols = [c for c in critical_columns if c in df.columns]
    null_report = df.select([
        spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(f"{c}_nulls")
        for c in existing_cols
    ]).collect()[0]

    all_passed = True
    for c in existing_cols:
        null_count = null_report[f"{c}_nulls"]
        null_pct = null_count / total_rows * 100
        status = "OK" if null_pct < 1 else "REVIEW"
        if null_pct >= 1:
            all_passed = False
        print(f"  [{status}] {c}: {null_count:,} nulls ({null_pct:.2f}%)")

    if "event_id" in df.columns:
        dup_count = total_rows - df.select("event_id").distinct().count()
        print(f"  [{'OK' if dup_count == 0 else 'REVIEW'}] duplicates: {dup_count:,}")

    print(f"Overall status: {'PASSED' if all_passed else 'NEEDS REVIEW'}")
    return all_passed

run_quality_checks(events_df, "silver_events", ["event_id", "user_id", "event_timestamp", "event_type"])


In [ ]:
print("--- Pattern 3: Slowly Changing Dimension (SCD Type 2) ---")

user_dimension = spark.createDataFrame([
    (1, "John Doe", "basic", "2024-01-01", "2024-01-15", False),
    (1, "John Doe", "premium", "2024-01-15", "9999-12-31", True),
    (2, "Jane Smith", "premium", "2024-01-01", "9999-12-31", True),
    (3, "Bob Wilson", "basic", "2024-01-01", "2024-02-01", False),
    (3, "Bob Wilson", "basic", "2024-02-01", "9999-12-31", True),
], ["user_id", "name", "tier", "valid_from", "valid_to", "is_current"])

user_dimension.show()
print("""
SCD Type 2: every change to a dimension row creates a NEW row with validity dates.
`is_current` marks the active version, enabling point-in-time queries while preserving full history.
""")


In [ ]:
print("--- Pattern 4: Safe Aggregation with error handling ---")

def safe_aggregate(df, metrics_config):
    results = {}
    for metric_name, cfg in metrics_config.items():
        try:
            filtered_df = df
            for f in cfg.get("filters", []):
                filtered_df = filtered_df.filter(f)
            result = filtered_df.agg(cfg["func"](cfg["column"])).collect()[0][0]
            results[metric_name] = result if result is not None else 0
        except Exception as e:
            print(f"Warning: failed to compute {metric_name}: {e}")
            results[metric_name] = None
    return results

metrics = {
    "total_events": {"func": count, "column": "*"},
    "unique_users": {"func": countDistinct, "column": "user_id"},
    "total_revenue": {"func": spark_sum, "column": "total_amount",
                       "filters": [col("event_type") == "purchase"]},
    "avg_order_value": {"func": avg, "column": "total_amount",
                         "filters": [col("event_type") == "purchase"]},
}

computed_metrics = safe_aggregate(events_df, metrics)
for name, value in computed_metrics.items():
    print(f"  {name}: {value:,.2f}" if isinstance(value, float) else f"  {name}: {value:,}")


## Lab 3 Summary

**Key concepts mastered**

- Window functions: `row_number`, `rank`, `dense_rank`, `ntile`, `lag`, `lead`, running totals, moving averages
- Partitioning strategies and partition pruning for query performance
- Caching/persistence and storage-level trade-offs
- Every join type (`inner`, `left`, `right`, `full`, `left_semi`, `left_anti`) plus broadcast joins
- Query optimization: filter pushdown, column pruning, reading execution plans
- UDFs vs pandas UDFs vs built-in functions, and when to use each
- A minimal Structured Streaming pipeline with a windowed aggregation
- Production patterns: incremental processing, data-quality checks, SCD Type 2, safe aggregation

Together, Lab 2 and Lab 3 cover the full journey of a Spark data-engineering pipeline: ingesting raw
data, cleaning and modeling it through the Medallion Architecture, and then analyzing, optimizing and
productionizing it.


In [ ]:
spark.stop()
print("Lab 3 complete! You now have an end-to-end view of building production Spark pipelines.")
